PART 1: `sample_config.yaml` -- The Competition Pipeline Config
   This file lives at `aic/aic_engine/config/sample_config.yaml`. It's the
   MASTER CONFIGURATION that `aic_engine` reads to set up and run competition
   trails. It defines EVERYTHING about what the scene looks like and what the
   robot needs to do.

SECTION 1: `scoring.topics` 
   This lists every ROS 2 topic the scoring system subscribes to. These are
   the signals the competition evaluator watches:

```yaml
scoring:
  topics:
    - topic: 
      name: "/joint_states"         # Where is each robot arm right now?
      type: "sensor_msgs/msg/JointState"
    - topic: 
      name: "/tf"
      ...
                                    # Live TF transforms (robot + gripper poses)
                                    
                                    # Cable link poses (where each cable segment
                                    # is)
                                    
                                    # Did you touch something forbidden?
                                    # Force/torque at the insertion point
                                    # What commands did your model send?
                                    # Cartesian pose commands
                                    # "SC plug inserted!" event
                                    # Controller internal state
```

   WHY THIS MATTERS: The evaluator records all of these during a trial. If your#
   robot touches an off-limit surface or exceed force limits, you lose points.
   The `insertion_event` topic is how the system knows a plug was successfully
   inserted.


---
Section 2: `task_board_limits`


```yaml
    min_translation: ...  # Minimum translation from center along rail...
    max_translations: ... # Maximum translation from center along rail (in meters)
```
   ...

   THIS IS THE LEGAL RANGE OF WHERE components can be placed along each rail
   type. The translation value is the OFFSET FROM RAIL CENTER ALONG THE Y-AXIS.
   This is what `randomize_board.py` should respect -- it defines the limit of 
   where NIC cards, SC ports, and mount fixtures can slide.


---
SECTION 3: `trails` -- The big one
   Each trail defines a COMPLETE SCENE ++ TASK. This is how the competition
   works. 

```yaml
trails:
  trail_1:                  <-- First scenario
    scene:                  <-- What the world looks like
      task_board:           <-- Board configuration
        pose:               <-- Where the board sits in the world
        nic_rail_0:         <-- What's on each rail
        ...
      cables:               <-- Cable assembly config
    tasks:                  <-- What the robot must accomplish
      task_1:               <-- insert this plug into that port    

```


Trail 1 scene breakdown:

```yaml
  task_board:
    pose:
      x: 0.15, y: -0.2, z: 1.14      # Board world position
      yaw: 3.1415                    # Rotated ~180 degree (facing the robot)

    nic_rail_0:
      entity_present: True           # There IS a NIC card here
      entity_name: "nic_card_0"      # Its name for TF/scoring
      entity_pose:
        translation: 0.036           # Slide 3.6cm from rail center
        yaw: 0.0                     # No rotation

    nic_rail_1 through 4:
      entity_present: False           # Empty rails

    sc_rail_0:
      entity_present: True
      entity_name: "sc_mount_0"       # NOTE: SC port on SC rail
      entity_pose:
        translation: 0.042
        yaw: 0.1                      # Slight way! Not always zero

    lc_mount_rail_0:
      entity_present: True
      entity_name: "lc_mount_0"       # LC mount fixture
      entity_pose:
        translation: 0.02

    sfp_mount_rail_0:
      entity_present: True
      entity_name: "sfp_mount_0"      # SFP mount fixture

    sc_mount_rail_0:
      entity_present: True
      entity_name: "sc_mount_0"       # SC mount fixture
```

   KEY INSIGHT YOU MENTIONED MISSING: The config shows that MOUNT FIXTURES CAN
   HAVE NON-ZERO YAW (`sc_rail_0` has `yaw: 0.1`). Our randomiser currently
   forces zero yaw -- which is correct for mounts (screwed to rails), but SC
   ports on SC rails CAN have slight yaw.


---
CABLE CONFIGURATION:
```yaml
        cables:                     
          cable_0:
            pose:
              gripper_offset:           # Where cable attaches to gripper
                x: 0.0
                y: 0.015385
                z: 0.04245
              roll: 0.4432
              pitch: -0.4838
              yaw: 1.3303
            attach_cable_to_gripper: True       # Cable starts held by robot
            cable_type: "sfp_sc_cable" # [sfp_sc_cable, sfp_sc_cable_reversed]
                                                # Which cable model (SFP on one end, SC on other)
```

   THIS IS CRITICAL: The cable is pre-attached to the gripper! The robot starts
   HOLDING the cable. The plug is at the cable end. This is why `aic_engine.cpp`
   doesn't spawn plugs separately -- the plug is physically connected to the 
   cable, which is held by the gripper.


---
TASK DEFINITION
```yaml
    tasks:
      task_1:
        cable_type: "sfp_sc"
        cable_name: "cable_0"
        plug_type: "sfp"
        plug_name: "sfp_tip"
        port_type: "sfp"
        port_name: "sfp_port_0"
        target_module_name: "nic_card_mount_0"
        time_limit: 180
```
   TRAIL 3 is different -- it changes the board pose (x=0.17, yaw=3.0), uses
   different rails, and tasks the robot with inserting the SC end 
   (`cable_type: "sfp_sc_cable_reversed"`) into an SC port.


---
SECTION 4: `robot` 
```yaml
robot:
  home_joint_positions:
    shoulder_pan_joint: -0.1597
    shoulder_lift_joint: -1.3542
    elbow_joint: -1.6648
    wrist_1_joint: -1.6933
    wrist_2_joint: 1.5710
    wrist_3_joint: 1.4110

```
   The robot's starting pose for each trail. This is where the UR5e arm begins
   before attempting insertion.



---
SUMMARY OF WHAT THIS CONFIG TELLS US

   1. The competition runs multiple trails with different board configurations
      and tasks
   2. Components slide along rails with bounded translation ranges
   3. The cable is pre-attached to the gripper -- plugs are cable endpoints, not
      standalone objects
   4. Mount fixtures CAN have slight yaw on SC rails (0.1 rad = 5.7 degree)
   5. Each trail has a specific insertion task -- insert plug X into port Y
      within time limit
   6. THE BOARD POSE CHANGES BETWEEN TRAILS -- the robot must handle different
      positions/orientations


---
PART 2: `scene_description.md + aic_bringup/README.md` -- The Official Pipeline
   These two docs describe the INTENDED WORKFLOW for setting up training
   environments. This is critical because WE'VE BEEN PARTIALLY REINVENTING THE
   WHEEL.

The Official Scene Pipeline (4 steps)
   
   1. CONFIGURE -> Launch params or `sample_config.yaml` define the scene
   2. SPAWN     -> Gazebo spawns task board + cable + robot via aic_engine
   3. EXPORT    -> World plugin auto-saves complete scene to `/tmp/aic.sdf`
   4. CONVERT   -> sdf2mjcf or Isaac Lab imports the SDF for training


KEY DISCOVERY #1: Gazebo exports the COMPLETE world state

From `scene_description.md`:

```bash
# Launch with a specific configuration
/entrypoint.sh spawn_task_board:=true nic_card_mount_2_present:=true \
    spawn_cable:=true cable_type:=sfp_sc_cable ground_truth:=true start_aic_engine:=false

# The COMPLETE world state is auto-saved to /tmp/aic.sdf
cp /tmp/aic.sdf ~/training_scenarios/nic_slot_2.sdf
```
   THIS MEANS: After Gazebo spawns everything (including physics-settled plugs
   in mounts), the entire scene -- with correct positions -- is saved as SDF.
   Then `sdf2mjcf` converts that to MJCF. The mount -> plug ALIGNMENT PROBLEM
   WE'VE BEEN FIGHTING SHOULDN'T EXIST IN A PROPERLY EXPORTED SCENE because 
   Gazebo already solved it via physics. 


---
KEY DISCOVERY #2: `ground_truth` TF frames
   From `aic_bringup/README.md`:
      `ground_truth` (default: "false") - Include ground truth pose data in TF
      topics.

   When `ground_truth:=true`, Gazebo publishes exact TF frames for every task 
   board component. This is exactly what the CHEATCODE POLICY needs -- and we
   noted it as a blocker ("needs GT TF frames"). We just need to launch Gazebo
    with this flag.


---
KEY DISCOVERY #3: The mount rail system is fully parameterised
   From `aic_bringup/README.md`... , every mount has 5 parameters:
      - `present` - is there a component here?
      - `translation` - where along the rail (Y offset, meters)
      - `roll`, `pitch`, `yaw` - orientation


THE 6 MOUNT RAILS:
        




In [ ]:

    The pipeline is:

    Gazebo SDF → sdf2mjcf → MJCF (scene.xml + aic_world.xml + OBJ meshes)
                                            ↓
                             Isaac Sim's built-in MJCF Importer
                             (isaacsim.asset.importer.mjcf extension)
                                            ↓
                                       USD scene file
                                            ↓
                                  Isaac Lab loads the USD

    The "intermediary" is Isaac Sim's MJCFCreateAsset command — it's a native C++ plugin (isaacsim.asset.importer.mjcf.plugin.dll) that:

    1. Parses the MJCF XML
    2. Reads all referenced OBJ meshes
    3. Converts the body hierarchy → USD prim hierarchy
    4. Converts geom transforms → USD Xform transforms
    5. Converts materials/textures → USD materials
    6. Converts joints/actuators → USD PhysX joints
    7. Outputs a USD stage in memory

    The key insight is that all the positions from the MJCF are preserved automatically because the importer reads them directly from the
    XML attributes (pos, quat on each body/geom). No manual calibration needed — the sdf2mjcf converter already put everything in the right
     place, and the MJCF importer just translates that to USD's coordinate system.

    The script I wrote (convert_mjcf_to_usd.py) calls this importer headlessly and saves the result as aic_world_from_mjcf.usd.

---

Part 3: `aic_utils/aic_isaac/` -- Every file explained

FILE STRUCTURE: 
```bash
  aic_isaac/
  ├── README.md                    ← Setup guide (Docker, assets, usage)
  └── aic_isaaclab/
      ├── scripts/                 ← Runnable scripts
      │   ├── random_agent.py      ← Runs random actions (for testing)
      │   ├── zero_agent.py        ← Sends zero actions (for testing)
      │   ├── list_envs.py         ← Lists registered environments
      │   ├── teleop.py            ← Keyboard/gamepad teleoperations
      │   ├── record_demos.py      ← Record teleop demos to HDF5
      │   ├── replay_demos.py      ← Replay recorded demos
      │   └── rsl_rl/
      │       ├── train.py         ← PPO training
      │       ├── play.py          ← Run trained policy
      │       └── cli_args.py      ← CLI argument parsing
      └── source/aic_task/aic_task/tasks/manager_based/aic_task/
          ├── __init__.py          ← Registers "AIC-Task-v0" in Gymnasium
          ├── aic_task_env_cfg.py  ← THE MAIN CONFIG (scene, MDP, everything)
          ├── agents/
          │   └── rsl_rl_ppo_cfg.py ← PPO hyperparameters
          ├── mdp/
          │   ├── __init__.py      ← Re-exports from Isaac Lab + custom functions
          │   ├── events.py        ← Domain randomisation (lights, board, parts)
          │   ├── observations.py  ← Contact force observation
          │   └── rewards.py       ← Position/orientation tracking rewards
          └── Intrinsic_assets/    ← NVIDIA-prepared USD assets
```

   `aic_task_env_cfg.py` -- The central config


SCENE (lines 50-252):
   * `robot` - UR5e loaded from `aic_unified_robot_cable_sdf.usd` at 
     pos=(-0.18, -0.122, 0). Arm actuators stiffness=2000.0, damping=100.0.
     Gripper commented out. 
   * `light` - Dome light at intensity 2500
   * `ground` - Ground plane at z=-1.05
   * `aic_scene` - Enclosure/floor from `scene/aic.usd` at z=-1.15
   * `task_board` - Board from `task_board_rigid.usd` at ... - kinematic
     (`kinematic_enabled=True`) (doesn't mvoe under physics)
   * `sc_port` - SC port at ... with rot=... (rotation)
   * `sc_port_2` - ...
   * `nic_card` ... NIC card at ...
   * 3 cameras (center/left/right) - 224x224 RGB ia TiledCamera on wrist


ACTIONS (line 589)     
   Differential IK, relative mode, SVD solver, scale=0.05


COMMANDS (line 264)
   Random EE pose targets in worksapce (x: 0.55 to 0.75; y: -0.1 to 0.02; 
   z: 0.01 to 0.15).



---
`events.py` -- DOMAIN RANDOMISATION





 aic_task_env_cfg.py — The central config

  Scene (lines 50-252):
  - robot — UR5e loaded from aic_unified_robot_cable_sdf.usd at pos=(-0.18, -0.122, 0). Arm actuators: stiffness=2000, damping=100. Gripper
  commented out.
  - light — Dome light at intensity 2500
  - ground — Ground plane at z=-1.05
  - aic_scene — Enclosure/floor from scene/aic.usd at z=-1.15
  - task_board — Board from task_board_rigid.usd at (0.2837, 0.229, 0.0) — kinematic (doesn't move under physics)
  - sc_port — SC port at (0.2904, 0.1928, 0.005) with rot=(0.73136, 0, 0, -0.682)
  - sc_port_2 — Second SC port at (0.2913, 0.1507, 0.005)
  - nic_card — NIC card at (0.25135, 0.25229, 0.0743)
  - 3 cameras (center/left/right) — 224x224 RGB via TiledCamera on wrist

  Key positions pattern: Each part is positioned as board_pos + offset:
  - SC port 1 offset: (0.0067, -0.0362, 0.005)
  - SC port 2 offset: (0.0076, -0.0783, 0.005)
  - NIC card offset: (-0.03235, 0.02329, 0.0743)

  Actions (line 589): Differential IK, relative mode, SVD solver, scale=0.05

  Commands (line 264): Random EE pose targets in workspace (x: 0.55-0.75, y: -0.1 to 0.02, z: 0.01-0.15)

  events.py — Domain randomization

  randomize_dome_light (line 113): Random intensity (1500-3500) and color per reset.

  randomize_board_and_parts (line 163): The key function. On reset:
  1. Moves board by random offset ±5mm in x and y
  2. For each part, applies its fixed offset from board + a random pose_range offset
  3. Caches orientations from first sim step — never recomputes them
  4. NIC card snaps to 4cm steps along Y (discrete rail positions)

  This is how NVIDIA handles positions — hardcoded offsets from board center, not derived from XACRO.

  rewards.py — Reward shaping

  Three layers of position tracking (coarse → fine → insertion):
  - position_command_error — L2 penalty (weight -0.2)
  - position_command_error_tanh — tanh kernel (weight 0.1, std=0.1)
  - position_command_error_exp — Gaussian kernel (weight 0.3, sigma=0.05)

  Orientation tracking:
  - orientation_command_error — angular distance penalty (weight -0.1)
  - orientation_command_error_tanh — tanh kernel (weight 0.05, std=0.25)

  Sparse bonus:
  - ee_reaching_bonus — +1 when within 2cm of target

  Smoothness: action rate, joint velocity, acceleration, torques, joint limits

  rsl_rl_ppo_cfg.py — Training hyperparameters

  - Network: [512, 256, 128] ELU for both actor and critic
  - 24 steps per env, 1500 max iterations
  - PPO with clip=0.2, entropy=0.006, adaptive LR starting at 1e-3
  - gamma=0.99, lambda=0.95

  observations.py — Custom observations

  Just one custom function: contact_net_forces — reads contact sensor data for selected bodies.

  The main observations are in the env config:
  - joint_pos (relative, with noise)
  - joint_vel (relative, with noise)
  - eef_pose (wrist_3_link world pose, 7D)
  - pose_command (target EE pose)
  - body_forces (wrenches on 7 links)
  - 3x camera RGB (through ResNet18 feature extractor)
  - last_action

  ---
  Now you have the full picture. Take your time reading through it. What stands out to you?

✻ Crunched for 1m 7s

❯ wait i dont get it... why cant we just directly use aic_task_env_cfg.py then? and does this file not like directly work with aic_engine
per se?

● Great question. Two separate things:

  1. aic_task_env_cfg.py already works. The Isaac Lab training (train.py, random_agent.py) runs fine with it. It has the board, SC ports,
  NIC card, robot — all correctly positioned by NVIDIA. We ran it earlier and it worked.

  2. It does NOT connect to aic_engine. They're completely independent systems:

  ┌───────────────┬────────────────────────────────────────┬─────────────────────────────────┐
  │               │               aic_engine               │ Isaac Lab (aic_task_env_cfg.py) │
  ├───────────────┼────────────────────────────────────────┼─────────────────────────────────┤
  │ Runs in       │ Gazebo (ROS 2)                         │ Isaac Sim (standalone)          │
  ├───────────────┼────────────────────────────────────────┼─────────────────────────────────┤
  │ Reads         │ sample_config.yaml                     │ Hardcoded Python config         │
  ├───────────────┼────────────────────────────────────────┼─────────────────────────────────┤
  │ Spawns        │ Task board + cable via ROS 2 services  │ USD assets via Isaac Lab API    │
  ├───────────────┼────────────────────────────────────────┼─────────────────────────────────┤
  │ Purpose       │ Competition evaluation + orchestration │ RL training + data collection   │
  ├───────────────┼────────────────────────────────────────┼─────────────────────────────────┤
  │ Randomization │ aic_engine reads trials from YAML      │ events.py randomizes per reset  │
  └───────────────┴────────────────────────────────────────┴─────────────────────────────────┘

  aic_engine is a ROS 2 node that talks to Gazebo. Isaac Lab is a completely separate Python framework that talks to PhysX directly. They
  don't communicate at all.

  So what's actually missing? The env config only has 3 component types (board, SC ports, NIC card). To add LC/SFP/SC mounts, you just need
  to:

  1. Add RigidObjectCfg entries with the right USD paths (already exist in Intrinsic_assets/assets/)
  2. Set correct positions (offsets from board center)
  3. Add them to the randomize_board_and_parts event

  The positions are the only hard part — and that's what NVIDIA hardcoded for the 3 existing components. For the mounts, we need to find the
   right offsets. That's the problem we've been going in circles on.

  Does this clarify things? The env config is the right file to edit — the question is just getting those 3 offset values for the mount
  fixtures.

---

-- In NVIDIA Isaac Sim (specifically within the Isaac Lab framework), a MARKOV
   DECISION PROCESS (MDP) is the mathematical framework used to define and 
   structure robot elarning tasks, such as reinforcement learning (RL).

   It acts as the foundation for how a robot interacts with its environment,
   breaking down the learning task into actionable components.

### Core Components of MDP in Isaac Lab
   Isaac lab maps the MDP framework directly to code, allowing users to define
   tasks through "Managers." The components include:

   * STATES/OBSERVATIONS (S): The data the agent (robot) receives to understand
     the current situation (e.g., joint positions, velocity, or sensory data).
   * ACTIONS (A): The decisions the agent can make, such as applying torque to
     joints or setting position targets.
   * REWARDS (R): A numerical value provided by the environment that encourages
     desirable behavior (e.g., moving forward) or discourages bad behavior
     (e.g., falling over).
   * TRANSITION PROBABILITY (T): The mechanism--physics engine in Isaac Sim--
     that determines the next state based on the current state and action.
   * TERMINATION: Rules that determine when an episode ends (e.g., robot falls,
     task is completed, or time limit exceeded).


### MDP IMPLEMENTATION IN ISAAC LAB
   Isaac Lab provides a MANAGER-BASED WORKFLOW to define these MDP termins in a
   modular and scalable way:

   1. OBSERVATION MANAGER: Handles what the agent sees.
   2. ACTION MANAGER: Processes policy outputs into motor commands.
   3. REWARD MANAGER: Defines the reward terms (e.g., `is_alive`, 
      `joint_pos_target_l2`).
   4. EVENT_MANAGER: Handles resets, randomisation, and environment changes.

   This structure is highly recommended for creating new environments in Isaac 
   Lab as it separates environment logic (actions, rewards) into reusable, 
   configurable Python classes.    

In [ ]:
❯ yeah but can't we extract the offsets from like aic_engine? or the MJCF files?

● Yes — from the MJCF, we can get exact board-relative positions for everything. Let me do that right now:

● Bash(wsl -e bash -lc '
      cd ~/projects/Project-Automaton/References/aic…)
  ⎿  Board pos: [ 0.15 -0.2   1.14]
     Board quat: [ 4.63111000e-05 -1.74943000e-08  3.66001000e-09  9.99999999e-01]
     Board R diagonal: (-1.000, -1.000, 1.000)
     … +30 lines (ctrl+o to expand)

● Here's where we are. The board rotation is confirmed as yaw=π (R diagonal = -1, -1, +1 — negates X and Y).

  The problem is clear now. The SC port correction is (-0.068, -0.007, -0.012) — this is the difference between where MuJoCo says the SC
  port is vs where Isaac Lab says it is. This correction accounts for the difference between the XACRO link origin and the USD model origin.

  But — this correction is specific to the SC port USD. The NIC card would give a completely different correction (-0.114, -0.151, +0.062)
  because the NIC card USD has a different internal origin.

  So the mount USDs (which the other Claude instance converted from GLB) will each have their own unknown internal origin offset. We can't
  reuse the SC port correction for them.

  The honest bottom line: Without knowing the internal origin of each mount USD model, we can only get approximate estimates. The estimates
  above (sfp: -0.123, 0.100, -0.002 etc.) use the SC port correction as a rough proxy. They'll be in the right neighborhood but probably off
   by 2-5cm.

  The cleanest fix would be to:
  1. Open each mount USD in Isaac Sim GUI
  2. Note its bounding box center
  3. Compute the offset from that
